In [ ]:
from PIL import Image, ImageDraw, ImageFont
import random
from wonderwords import RandomWord

In [ ]:
_rw = RandomWord()

In [ ]:
def _random_word(length, rng=random):
    """Английское слово нужной длины или None."""
    if length <= 0:
        return None
    try:
        return _rw.word(word_min_length=length, word_max_length=length)
    except Exception:
        return None


def _rus_letters(length, alphabet, rng=random):
    return "".join(rng.choice(alphabet) for _ in range(length))


def _pick_segments(mask_px, row, width):
    segs = []
    c = 0
    while c < width:
        if mask_px[c, row] == 0:
            c += 1
            continue
        s = c
        while c < width and mask_px[c, row] != 0:
            c += 1
        segs.append((s, c))
    return segs


def fill_form_by_mask(
    background_path,
    mask_path,
    font_path,
    output_path,
    x0, y0, dx, dy, w, h,
    alphabet="АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ0123456789",
    threshold=128,
    jitter=2,
    font_size=None,
    text_color=(0, 0, 0, 255),
    use_words=True, 
    word_alphabet="ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    max_word_len=12,
):
    img = Image.open(background_path).convert("RGBA")
    draw = ImageDraw.Draw(img)

    mask = Image.open(mask_path).convert("L")
    mask = mask.resize((w, h), resample=Image.NEAREST)
    mask = mask.point(lambda p: 255 if p < threshold else 0)
    mask_px = mask.load()

    if font_size is None:
        font_size = int(dy * 0.85)
    font = ImageFont.truetype(font_path, size=font_size)

    for row in range(h):
        segments = _pick_segments(mask_px, row, w)

        for (s, e) in segments:
            L = e - s
            if not use_words:
                chars = _rus_letters(L, alphabet)
            else:
                chars = _word_or_fallback(
                    L, word_alphabet, alphabet,
                    max_word_len=max_word_len,
                )

            for i, ch in enumerate(chars):
                col = s + i
                cell_x = x0 + col * dx
                cell_y = y0 + row * dy

                bbox = draw.textbbox((0, 0), ch, font=font)
                tw = bbox[2] - bbox[0]
                th = bbox[3] - bbox[1]
                pos_x = cell_x + (dx - tw) / 2 - bbox[0] + random.randint(-jitter, jitter)
                pos_y = cell_y + (dy - th) / 2 - bbox[1] + random.randint(-jitter, jitter)

                draw.text((pos_x, pos_y), ch, font=font, fill=text_color)

    img.convert("RGB").save(output_path)


def _word_or_fallback(L, word_alphabet, rus_alphabet, max_word_len):
    if L <= 0:
        return ""

    if L <= max_word_len:
        w = _random_word(L)
        if w:
            return w.upper()
        return _rus_letters(L, rus_alphabet)

    parts = []
    remaining = L
    while remaining > 0:
        chunk = min(remaining, max_word_len)
        w = _random_word(chunk)
        if w:
            parts.append(w.upper())
        else:
            parts.append(_rus_letters(chunk, rus_alphabet))
        remaining -= chunk
    return "".join(parts)

In [ ]:
fill_form_by_mask(
    background_path="blank_2.jpg",
    mask_path="mask.jpg",
    font_path="Caveat-VariableFont_wght.ttf",
    output_path="output.jpg",
    x0=46, y0=377,
    dx=29.3, dy=29.3,
    w=39, h=44,
    threshold=128,
    jitter=1,
    font_size=26,
    use_words=True
)